In [1]:

import collections
import gzip
import glob
import os
import json
import warnings
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from scipy.signal import find_peaks, savgol_filter
from matplotlib.dates import DateFormatter, HourLocator
import matplotlib.dates as mdates
import matplotlib as mpl
from matplotlib.font_manager import FontProperties
from datetime import timedelta, datetime

warnings.filterwarnings("ignore")
print("DONE")

DONE


In [2]:
# 数据处理
data_path = 'exports'

In [3]:
zxswfile_pattern = "*闸下潮位.csv"
files = glob.glob(f"{data_path}/**/{zxswfile_pattern}", recursive=True)
if not files:
    raise FileNotFoundError(f"未找到匹配的文件: {zxswfile_pattern}")

zxswdf_list = []
for f in files:
    # 读取CSV，无表头
    temp_df = pd.read_csv(f, header=None, names=['station_id', 'time', 'water_level'])
    zxswdf_list.append(temp_df)

zxsw_df = pd.concat(zxswdf_list, ignore_index=True)
print("size:", zxsw_df.shape)

size: (32119, 3)


In [4]:
zxsw_df.head(10)
zxsw_df.to_csv("imports_v/闸下潮位.csv", index=False)

In [5]:
llfile_pattern = "*流量.csv"
files = glob.glob(f"{data_path}/**/{llfile_pattern}", recursive=True)
if not files:
    raise FileNotFoundError(f"未找到匹配的文件: {llfile_pattern}")

lldf_list = []
for f in files:
    # 读取CSV，无表头
    temp_df = pd.read_csv(f)
    lldf_list.append(temp_df)

ll_df = pd.concat(lldf_list, ignore_index=True)
print("size:", ll_df.shape)

size: (15643, 3)


In [6]:
ll_df.to_csv("imports_v/实测流量.csv", index=False)
ll_df.head(10)


,测点编码,监测日期,流量
0,8164,2021-01-11 10:00:00,0.00
1,8164,2021-01-11 11:00:00,11.93
2,8164,2021-01-11 12:00:00,3.58
3,8164,2021-01-11 13:00:00,0.00
4,8164,2021-01-11 14:00:00,0.00
5,8164,2021-01-11 15:00:00,16.77
6,8164,2021-01-11 16:00:00,5.99
7,8164,2021-01-11 17:00:00,46.72
8,8164,2021-01-11 18:00:00,7.19
9,8164,2021-01-11 19:00:00,9.58


In [7]:
scjyfile_pattern = "*实测降雨.csv"
files = glob.glob(f"{data_path}/**/{scjyfile_pattern}", recursive=True)
if not files:
    raise FileNotFoundError(f"未找到匹配的文件: {scjyfile_pattern}")

scjydf_list = []
for f in files:
    # 读取CSV，无表头
    temp_df = pd.read_csv(f)
    scjydf_list.append(temp_df)

scjy_df = pd.concat(scjydf_list, ignore_index=True)
print("size:", scjy_df.shape)

size: (19062379, 6)


In [8]:
scjy_df.to_csv("imports_v/实测降雨.csv", index=False)
scjy_df.head(10)

,测点编码,监测日期,雨量,经度,维度,所属区域
0,5427,2021-01-11 09:10:00,0.0,120.615600,29.765100,虞南山区
1,5425,2021-01-11 09:10:00,0.0,NaN,NaN,NaN
2,5422,2021-01-11 09:10:00,0.0,120.741999,29.861333,虞南山区
3,1532,2021-01-11 09:10:00,0.0,120.500000,29.894700,绍兴平原
4,3642,2021-01-11 09:10:00,0.0,120.698655,30.276754,NaN
5,3228,2021-01-11 09:10:00,0.0,120.598589,29.615754,嵊州
6,3840,2021-01-11 09:10:00,0.0,120.844937,29.917734,虞北平原
7,2310,2021-01-11 09:10:00,0.0,120.927700,30.059000,虞北平原
8,1601,2021-01-11 09:10:00,0.0,121.190897,29.523922,嵊州
9,7308,2021-01-11 09:10:00,0.0,120.744015,30.232381,NaN


In [9]:
scjy_df['所属区域'].unique()

array(['虞南山区', nan, '绍兴平原', '嵊州', '虞北平原', '新昌'], dtype=object)

In [10]:
import re
def parse_time(time_str):
    # 处理示例中的异常格式
    if isinstance(time_str, str) and len(time_str) > 26:
        # 策略1: 提取第一个完整的时间戳
        match = re.search(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})', time_str)
        if match:
            return pd.to_datetime(match.group(1))
        
        # 策略2: 分割为两个时间点并取平均值
        try:
            part1 = time_str[:26]  # 取前26个字符
            part2 = time_str[26:52]  # 取后26个字符
            dt1 = pd.to_datetime(part1)
            dt2 = pd.to_datetime(part2)
            # 取两个时间点的中间值
            return dt1 + (dt2 - dt1)/2
        except:
            return pd.NaT
    else:
        return pd.to_datetime(time_str)

In [11]:
jyybfile_pattern = "*降雨预报.csv"
files = glob.glob(f"{data_path}/**/{jyybfile_pattern}", recursive=True)
if not files:
    raise FileNotFoundError(f"未找到匹配的文件: {jyybfile_pattern}")

yyybdf_list = []
for f in files:
    try:
        # 读取文件，使用header=None保持原格式
        temp_df = pd.read_csv(f)
        # 检查DataFrame是否为空
        if temp_df.empty:
            continue  # 跳过空文件
        
        yyybdf_list.append(temp_df)
    
    except Exception as e:
        # print(f"文件 {f} 读取错误: {str(e)}")
        pass

jyyb_df = pd.concat(yyybdf_list, ignore_index=True)
print("size:", jyyb_df.shape)

size: (291971, 7)


In [12]:
jyyb_df['时间'] = jyyb_df['时间'].apply(parse_time)

DateParseError: Unknown datetime string format, unable to parse: 预计结束时间, at position 0

In [ ]:
jyyb_df['所属区域'].unique()

array(['绍兴平原', '嵊州', '虞南山区', '新昌', '虞北平原'], dtype=object)

In [ ]:
jyyb_df.head(10)

,经度,纬度,时间,雨量,市,区,所属区域
0,120.25,30.15,2025-04-23 14:00:00,0.0,NaN,NaN,绍兴平原
1,120.35,30.05,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
2,120.35,30.15,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
3,120.45,29.95,2025-04-23 14:00:00,0.1,绍兴市,柯桥区,绍兴平原
4,120.45,30.05,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
5,120.45,30.15,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
6,120.55,29.35,2025-04-23 14:00:00,0.0,NaN,NaN,嵊州
7,120.55,29.45,2025-04-23 14:00:00,0.0,绍兴市,嵊州市,嵊州
8,120.55,29.55,2025-04-23 14:00:00,0.0,绍兴市,嵊州市,嵊州
9,120.55,29.65,2025-04-23 14:00:00,0.0,绍兴市,嵊州市,嵊州


In [ ]:
jyyb_df.to_csv("imports_v/降雨预报.csv", index=False)
jyyb_df.head(10)

,经度,纬度,时间,雨量,市,区,所属区域
0,120.25,30.15,2025-04-23 14:00:00,0.0,NaN,NaN,绍兴平原
1,120.35,30.05,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
2,120.35,30.15,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
3,120.45,29.95,2025-04-23 14:00:00,0.1,绍兴市,柯桥区,绍兴平原
4,120.45,30.05,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
5,120.45,30.15,2025-04-23 14:00:00,0.0,绍兴市,柯桥区,绍兴平原
6,120.55,29.35,2025-04-23 14:00:00,0.0,NaN,NaN,嵊州
7,120.55,29.45,2025-04-23 14:00:00,0.0,绍兴市,嵊州市,嵊州
8,120.55,29.55,2025-04-23 14:00:00,0.0,绍兴市,嵊州市,嵊州
9,120.55,29.65,2025-04-23 14:00:00,0.0,绍兴市,嵊州市,嵊州


In [ ]:
swgkfile_pattern = "*水位工况.csv"
files = glob.glob(f"{data_path}/**/{swgkfile_pattern}", recursive=True)
if not files:
    raise FileNotFoundError(f"未找到匹配的文件: {swgkfile_pattern}")

swgkdf_list = []
for f in files:
    # try:
    #     # 读取文件，使用header=None保持原格式
    #     temp_df = pd.read_csv(f)
    #     if temp_df.empty:
    #         continue  # 跳过空文件
    #     swgkdf_list.append(temp_df)
    # except Exception as e:
    #     print(f"文件 {f} 读取错误: {str(e)}")
    names = ["测点编码","监测日期","水位"]
    with open(f, 'r') as file:
        for i, line in enumerate(file.readlines()):
            if i == 0:
                pass
            else:
                tokens = line.strip().split(',')
                if len(tokens) == len(names):
                    swgkdf_list.append(dict(zip(names, tokens)))

print("size:", len(swgkdf_list))
# swgk_df = pd.concat(swgkdf_list, ignore_index=True)
swgk_df = pd.DataFrame(swgkdf_list)
# swgk_df.to_csv("水位工况.csv", index=False)
print("size:", swgk_df.shape)

size: 226539
size: (226539, 3)


In [ ]:
swgk_df = swgk_df.dropna()
swgk_df = swgk_df.drop_duplicates()

# 1. 将 'value' 列转换为数值类型，非数字转为 NaN
swgk_df['水位'] = pd.to_numeric(swgk_df['水位'], errors='coerce')

# 2. 删除 'value' 列中为 NaN 的行（即原非数字数据所在行）
swgk_df = swgk_df.dropna(subset=['水位'])
swgk_df.to_csv("imports_v/水位工况.csv", index=False)
swgk_df.head(10)

,测点编码,监测日期,水位
412,1529,2021-01-11 10:00:00,4.09
413,1529,2021-01-11 11:00:00,4.08
414,1529,2021-01-11 12:00:00,4.08
415,1529,2021-01-11 13:00:00,4.08
416,1529,2021-01-11 14:00:00,4.08
417,1529,2021-01-11 15:00:00,4.08
418,1529,2021-01-11 16:00:00,4.09
419,1529,2021-01-11 17:00:00,4.09
420,1529,2021-01-11 18:00:00,4.09
421,1529,2021-01-11 19:00:00,4.10


In [ ]:
dlpath = "data/调令信息.csv"
dl_df = pd.read_csv(dlpath)
dl_df.head(10)

,SIGNTM,日期,调度信息,开闸时间,关闸时间,开闸时长,开闸孔数,目标水位
0,2025-07-30 16:00:11,2025-07-30,07月30日傍晚退潮时段开启28孔闸门，到潮前关闭全部闸门，预计涨潮时间07-31 01:40,NaN,NaN,NaN,28,无最低水位限制
1,2025-07-30 05:55:49,2025-07-30,07月30日上午退潮时段开启28孔闸门，到潮前关闭全部闸门，预计涨潮时间07-30 13:45,07:15,13:10,NaN,28,无最低水位限制
2,2025-07-29 18:44:19,2025-07-29,07月29日傍晚退潮时段开启16孔闸门，07-30 00:30关闭全部闸门，预计涨潮时间07...,19:20,00:30,NaN,16,无最低水位限制
3,2025-07-29 05:54:14,2025-07-29,07月29日上午退潮时段开启20孔闸门，07-29 12:30关闭全部闸门，预计涨潮时间07...,06:30,12:30,NaN,20,无最低水位限制
4,2025-07-28 16:01:16,2025-07-28,07月28日傍晚退潮时段开启20孔闸门，07-29 00:00关闭全部闸门，预计涨潮时间07...,17:40,00:00,NaN,20,无最低水位限制
5,2025-07-28 07:08:13,2025-07-28,07月28日上午退潮时段开启12孔闸门，开闸时间3小时，预计涨潮时间07-28 13:00,08:00,11:00,3.0,12,2.5
6,2025-07-27 14:00:41,2025-07-27,07月27日下午退潮时段开启20孔闸门，开闸时间5小时，预计涨潮时间07-27 23:55,17:00,22:00,5.0,20,2.00
7,2025-07-25 14:57:03,2025-07-25,07月25日下午退潮时段开启14孔闸门，开闸时间3小时，预计涨潮时间07-25 22:50,15:40,18:40,3.0,14,3.00
8,2025-07-19 07:55:02,2025-07-19,07月19日上午退潮时段开启6孔闸门，开闸时间3小时，预计涨潮时间07-19 16:00,08:30,11:30,3.0,6,3.40
9,2025-07-18 20:32:49,2025-07-18,07月18日晚上退潮时段开启14孔闸门，开闸时间3小时，预计涨潮时间07-19 03:30,21:20,00:20,3.0,14,3.0
